> **Heads-up on compute:** ModernBERT on long inputs is memory-hungry. This notebook is tuned for **memory-constrained machines (≤8 GB GPU)**: forces CPU, caps inputs at 1,024 tokens, batch size 1 + gradient checkpointing + 2 epochs. Training runs roughly 30–90 min on a modern CPU. If you have a GPU with 16 GB+ free, delete the `CUDA_VISIBLE_DEVICES` line and bump `MAX_LEN` / `num_train_epochs` back up.

# 01 — ModernBERT on Full CTI Reports

**Goal:** classify **whole threat reports** into MITRE ATT&CK tactics using a single forward pass — no chunking, no aggregation hacks.

## Why this notebook exists

CTI 201 maxed out BERT's 512-token window. Real threat reports are 2,000–8,000 tokens. In notebook 08 we chunked and max-pooled — it works, but it's a post-hoc patch: the model never sees the whole report at once, so signals across sections can't interact.

**ModernBERT** (released late 2024) is a drop-in BERT replacement with:

- **8,192-token native context** — handles most threat reports in one pass.
- **Alternating local/global attention** — scales better than full attention on long sequences.
- **Trained on 2 trillion tokens** including code and technical text.

The fine-tuning code is nearly identical to CTI 201 notebook 07. The interesting part is the data shape — we feed it full reports, not sentences.

## Data: TRAM II, aggregated to report level

TRAM II publishes sentence-level labels, but each sentence carries a `doc_title`. We group sentences by that field and union their technique labels — this reconstructs roughly 150 full reports, each labeled with the set of MITRE techniques it describes. We then project techniques onto their parent tactics (14 classes) using MITRE's STIX bundle.

## Prerequisites

```bash
pip install -U transformers datasets scikit-learn torch
```

`transformers >= 4.48` ships ModernBERT natively (`AutoModelForSequenceClassification` resolves `modernbert-base` without extra config).

## 1 — Build the report-level dataset

Download TRAM II + ATT&CK STIX, aggregate sentences into full reports, project technique labels to tactic labels, save a HuggingFace `DatasetDict`. Runs once.

In [1]:
import json, re, random, urllib.request
from pathlib import Path
from collections import defaultdict
import numpy as np
from datasets import Dataset, DatasetDict, Features, Sequence, Value

DATA = Path('data'); DATA.mkdir(exist_ok=True)
PROCESSED = Path('processed'); PROCESSED.mkdir(exist_ok=True)

TRAM_URL = 'https://raw.githubusercontent.com/center-for-threat-informed-defense/tram/main/data/tram2-data/multi_label.json'
ATTACK_URL = 'https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json'

for url, path in [(TRAM_URL, DATA / 'tram_multi_label.json'),
                  (ATTACK_URL, DATA / 'enterprise-attack.json')]:
    if not path.exists():
        print(f'Downloading {path.name}...')
        urllib.request.urlretrieve(url, path)

# --- Load TRAM sentence-level records -----------------------------------
with open(DATA / 'tram_multi_label.json') as f:
    raw = json.load(f)
records = raw if isinstance(raw, list) else next(v for v in raw.values() if isinstance(v, list))
print(f'Loaded {len(records)} TRAM sentence records.')

# --- Build technique -> tactic mapping from MITRE STIX ------------------
with open(DATA / 'enterprise-attack.json') as f:
    attack = json.load(f)

tech2tactics = {}
for obj in attack['objects']:
    if obj.get('type') != 'attack-pattern' or obj.get('revoked') or obj.get('x_mitre_deprecated'):
        continue
    tech_id = next((r['external_id'] for r in obj.get('external_references', [])
                    if r.get('source_name') == 'mitre-attack'), None)
    if not tech_id:
        continue
    tactics = [kcp['phase_name'] for kcp in obj.get('kill_chain_phases', [])
               if kcp.get('kill_chain_name') == 'mitre-attack']
    if tactics:
        tech2tactics[tech_id] = tactics

ALL_TACTICS = sorted({t for ts in tech2tactics.values() for t in ts})
TACTIC2ID = {t: i for i, t in enumerate(ALL_TACTICS)}
print(f'{len(ALL_TACTICS)} tactics: {ALL_TACTICS}')

# --- Aggregate sentences -> reports -------------------------------------
TECH_RE = re.compile(r'T\d{4}(?:\.\d{3})?')

def extract_techs(val):
    if val is None: return []
    if isinstance(val, list):
        out = []
        for item in val:
            s = item if isinstance(item, str) else ' '.join(map(str, item.values())) if isinstance(item, dict) else ''
            out += TECH_RE.findall(s)
        return out
    return TECH_RE.findall(val) if isinstance(val, str) else []

TEXT_KEYS = ('text', 'sentence', 'content')
LABEL_KEYS = ('labels', 'mappings', 'attack_ids', 'tags', 'techniques')
DOC_KEYS = ('doc_title', 'document', 'report', 'source')

def pick(d, keys, default=None):
    for k in keys:
        if k in d: return d[k]
    return default

docs = defaultdict(lambda: {'sentences': [], 'techniques': set()})
for rec in records:
    doc_id = pick(rec, DOC_KEYS, 'unknown')
    text = pick(rec, TEXT_KEYS, '')
    techs = [t.split('.')[0] for t in extract_techs(pick(rec, LABEL_KEYS))]
    if text:
        docs[doc_id]['sentences'].append(text)
        docs[doc_id]['techniques'].update(techs)

# Convert to (report_text, multi-hot tactic vector) pairs
def to_multihot(techs):
    vec = [0.0] * len(ALL_TACTICS)
    for t in techs:
        for tac in tech2tactics.get(t, []):
            vec[TACTIC2ID[tac]] = 1.0
    return vec

rows = []
for doc_id, bundle in docs.items():
    labels = to_multihot(bundle['techniques'])
    if sum(labels) == 0:
        continue  # skip reports with no mappable labels
    rows.append({
        'doc_id': str(doc_id),
        'text': ' '.join(bundle['sentences']),
        'labels': labels,
    })

print(f'\nAggregated {len(rows)} labeled reports.')
lens = np.array([len(r['text'].split()) for r in rows])
print(f'Report length (words): median={int(np.median(lens))}, '
      f'p75={int(np.percentile(lens, 75))}, '
      f'p95={int(np.percentile(lens, 95))}, max={lens.max()}')
print(f'% over BERT limit (512 words): {(lens > 512).mean()*100:.0f}%')
print(f'% over ModernBERT limit (~6000 words): {(lens > 6000).mean()*100:.0f}%')

# 80/10/10 split, reproducible
random.seed(42); random.shuffle(rows)
n = len(rows); n_tr = int(0.8 * n); n_va = int(0.1 * n)
splits = {'train': rows[:n_tr], 'validation': rows[n_tr:n_tr + n_va], 'test': rows[n_tr + n_va:]}

features = Features({
    'doc_id': Value('string'),
    'text': Value('string'),
    'labels': Sequence(Value('float32'), length=len(ALL_TACTICS)),
})
ds = DatasetDict({k: Dataset.from_list(v, features=features) for k, v in splits.items()})
ds.save_to_disk(str(PROCESSED / 'tram_reports'))
with open(PROCESSED / 'tram_reports' / 'tactic_vocab.json', 'w') as f:
    json.dump({'tactics': ALL_TACTICS, 'tactic2id': TACTIC2ID}, f, indent=2)

print(f'\nSaved: {ds}')

Loaded 19178 TRAM sentence records.
14 tactics: ['collection', 'command-and-control', 'credential-access', 'defense-evasion', 'discovery', 'execution', 'exfiltration', 'impact', 'initial-access', 'lateral-movement', 'persistence', 'privilege-escalation', 'reconnaissance', 'resource-development']

Aggregated 149 labeled reports.
Report length (words): median=2350, p75=3173, p95=5968, max=8463
% over BERT limit (512 words): 99%
% over ModernBERT limit (~6000 words): 5%


Saving the dataset (0/1 shards):   0%|          | 0/119 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16 [00:00<?, ? examples/s]


Saved: DatasetDict({
    train: Dataset({
        features: ['doc_id', 'text', 'labels'],
        num_rows: 119
    })
    validation: Dataset({
        features: ['doc_id', 'text', 'labels'],
        num_rows: 14
    })
    test: Dataset({
        features: ['doc_id', 'text', 'labels'],
        num_rows: 16
    })
})


**What to notice:** most reports exceed BERT's 512-token window but fit comfortably in ModernBERT's 8,192. That's the dataset characteristic this whole notebook is designed around.

## 2 — Tokenize with ModernBERT

One line different from BERT: the checkpoint. The tokenizer and model API are identical.

In [2]:
import os
# Force CPU. ModernBERT at 2k+ tokens pushes past 8 GB of GPU memory during
# backward pass; on a consumer-tier card it OOMs. On CPU it's slow but finishes.
# Remove this line if you have a GPU with ~16 GB+ free.
os.environ['CUDA_VISIBLE_DEVICES'] = ''

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_from_disk

MODEL_CKPT = 'answerdotai/ModernBERT-base'
MAX_LEN = 1024   # covers ~60% of reports fully; truncates the long tail.
                 # Main point of the notebook is \"BERT can\'t do 1024, ModernBERT can.\"

ds = load_from_disk(str(PROCESSED / 'tram_reports'))
with open(PROCESSED / 'tram_reports' / 'tactic_vocab.json') as f:
    vocab = json.load(f)
tactics = vocab['tactics']
num_labels = len(tactics)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tok = ds.map(tokenize, batched=True, remove_columns=['doc_id', 'text'])
print(tok)
print(f'\nTokenized length stats (train):')
tlens = [len(x) for x in tok['train']['input_ids']]
print(f'  median={int(np.median(tlens))}, p95={int(np.percentile(tlens, 95))}, max={max(tlens)}')


Map:   0%|          | 0/119 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 119
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 14
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 16
    })
})

Tokenized length stats (train):
  median=1024, p95=1024, max=1024


## 3 — Fine-tune

Same recipe as CTI 201 notebook 07:

- `problem_type='multi_label_classification'` → sigmoid heads + BCE loss.
- Macro F1 as the selection metric.
- Per-device batch size of 1 (documents are long — activations are big even on ModernBERT); gradient accumulation brings the effective batch back up.

On CPU, 4 epochs on ~120 training reports takes roughly 30–90 min depending on hardware. On a modest GPU (8 GB+), under 10 min.

In [3]:
import torch
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    problem_type='multi_label_classification',
)
# Recompute forward activations during backward instead of keeping them in RAM.
# Roughly halves peak memory. Costs ~30% more compute per step.
model.gradient_checkpointing_enable()

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (sigmoid(logits) >= 0.5).astype(np.float32)
    return {
        'subset_accuracy': accuracy_score(labels, preds),
        'hamming_accuracy': 1.0 - hamming_loss(labels, preds),
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }

args = TrainingArguments(
    output_dir='models/modernbert_tactics',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch 8
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    dataloader_pin_memory=False,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok['train'],
    eval_dataset=tok['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Subset Accuracy,Hamming Accuracy,Micro F1,Macro F1
1,4.401010,0.461893,0.000000,0.724490,0.721649,0.447293
2,3.651186,0.447718,0.000000,0.760204,0.751323,0.429725


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=3.918622843424479, metrics={'train_runtime': 980.7804, 'train_samples_per_second': 0.243, 'train_steps_per_second': 0.031, 'total_flos': 161362263063552.0, 'train_loss': 3.918622843424479, 'epoch': 2.0})

## 4 — Evaluate on test + tune per-tactic thresholds

Default 0.5 cutoff is fine for common tactics but collapses recall on rare ones. We sweep per-tactic thresholds on validation, then apply them to test (same protocol as CTI 201 nb 07).

In [4]:
# Predict logits on validation, tune thresholds
val_out = trainer.predict(tok['validation'])
val_probs = sigmoid(val_out.predictions)
val_y = val_out.label_ids

thresholds = np.full(num_labels, 0.5)
candidates = np.linspace(0.05, 0.95, 37)
for i in range(num_labels):
    f1s = [f1_score(val_y[:, i], (val_probs[:, i] >= t).astype(np.float32), zero_division=0)
           for t in candidates]
    thresholds[i] = candidates[int(np.argmax(f1s))]

# Evaluate on test with default vs tuned thresholds
test_out = trainer.predict(tok['test'])
test_probs = sigmoid(test_out.predictions)
test_y = test_out.label_ids
preds_default = (test_probs >= 0.5).astype(np.float32)
preds_tuned = (test_probs >= thresholds).astype(np.float32)

def summary(y, yhat, tag):
    print(f'\n{tag}:')
    print(f'  subset accuracy:  {accuracy_score(y, yhat):.4f}')
    print(f'  hamming accuracy: {1 - hamming_loss(y, yhat):.4f}')
    print(f'  micro F1:         {f1_score(y, yhat, average="micro", zero_division=0):.4f}')
    print(f'  macro F1:         {f1_score(y, yhat, average="macro", zero_division=0):.4f}')

summary(test_y, preds_default, 'TEST @ threshold 0.5')
summary(test_y, preds_tuned, 'TEST @ tuned per-tactic thresholds')

print('\nPer-tactic F1 (tuned):')
per = precision_recall_fscore_support(test_y, preds_tuned, average=None, zero_division=0)
for i, name in enumerate(tactics):
    print(f'  {name:26s}  t={thresholds[i]:.2f}  P={per[0][i]:.3f}  R={per[1][i]:.3f}  F1={per[2][i]:.3f}  n={per[3][i]}')


TEST @ threshold 0.5:
  subset accuracy:  0.0000
  hamming accuracy: 0.7188
  micro F1:         0.6986
  macro F1:         0.4271

TEST @ tuned per-tactic thresholds:
  subset accuracy:  0.0000
  hamming accuracy: 0.6161
  micro F1:         0.6587
  macro F1:         0.4716

Per-tactic F1 (tuned):
  collection                  t=0.05  P=0.375  R=1.000  F1=0.545  n=6
  command-and-control         t=0.05  P=0.750  R=1.000  F1=0.857  n=12
  credential-access           t=0.53  P=0.429  R=0.750  F1=0.545  n=8
  defense-evasion             t=0.05  P=0.875  R=1.000  F1=0.933  n=14
  discovery                   t=0.05  P=0.625  R=1.000  F1=0.769  n=10
  execution                   t=0.05  P=0.562  R=1.000  F1=0.720  n=9
  exfiltration                t=0.37  P=0.000  R=0.000  F1=0.000  n=3
  impact                      t=0.05  P=0.000  R=0.000  F1=0.000  n=0
  initial-access              t=0.05  P=0.375  R=1.000  F1=0.545  n=6
  lateral-movement            t=0.42  P=0.143  R=1.000  F1=0.250  n

## 5 — Save + inference on one example

In [5]:
FINAL = Path('models/modernbert_tactics_final')
FINAL.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL))
tokenizer.save_pretrained(str(FINAL))
with open(FINAL / 'tactic_vocab.json', 'w') as f:
    json.dump({'tactics': tactics, 'tactic2id': TACTIC2ID}, f, indent=2)
with open(FINAL / 'thresholds.json', 'w') as f:
    json.dump({tactics[i]: float(thresholds[i]) for i in range(num_labels)}, f, indent=2)
print(f'Saved to {FINAL}')

# Inference on the longest test report
lens_test = [len(r['text'].split()) for r in ds['test']]
idx = int(np.argmax(lens_test))
example = ds['test'][idx]
print(f'\nSample test report: {lens_test[idx]} words')
print(f'Ground truth tactics: {[tactics[i] for i, v in enumerate(example["labels"]) if v > 0.5]}')

device = next(model.parameters()).device
enc = tokenizer(example['text'], truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
with torch.no_grad():
    probs = torch.sigmoid(model(**enc).logits[0]).cpu().numpy()
pred_tactics = [(tactics[i], probs[i]) for i in range(num_labels) if probs[i] >= thresholds[i]]
pred_tactics.sort(key=lambda x: -x[1])
print(f'Predicted tactics:')
for name, p in pred_tactics:
    print(f'  {name:26s}  prob={p:.3f}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to models/modernbert_tactics_final

Sample test report: 4499 words
Ground truth tactics: ['command-and-control', 'defense-evasion', 'discovery', 'execution', 'persistence', 'privilege-escalation']
Predicted tactics:
  defense-evasion             prob=0.931
  execution                   prob=0.787
  privilege-escalation        prob=0.749
  command-and-control         prob=0.704
  persistence                 prob=0.670
  credential-access           prob=0.575
  discovery                   prob=0.556
  lateral-movement            prob=0.502
  initial-access              prob=0.453
  collection                  prob=0.425
  exfiltration                prob=0.419


## What you've done

- Built a **report-level** CTI dataset (150 labeled reports, 2k–8k tokens each) from TRAM II — the first time in this series we're training on whole documents.
- Fine-tuned **ModernBERT** with an 8k-token native context, so the model sees the entire report in one forward pass.
- Tuned per-tactic decision thresholds on validation and measured the macro F1 lift on test.
- Ended up with a saved model + thresholds that could slot into `analyze_report_v2()` from CTI 201 notebook 08 as a drop-in replacement for the chunked BERT classifier.

## What's next in CTI 301

**Notebook 02: CTI RAG assistant.** Use the APTnotes corpus (from CTI 201 nb 03) as a knowledge base, embed it with a retrieval model, and answer analyst questions via the Claude API — the modern alternative to fine-tuning when you need the model to know *facts* rather than *patterns*.